In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from collections import defaultdict

In [ ]:
def has_flag(arr, bit):
    return (arr & (1 << bit)) != 0

In [ ]:
# -----------------------------
# USER INPUT
# -----------------------------
data_folder = r"../data/OUTarea2\Jul"


In [ ]:
#checking how many scenes per year

from pathlib import Path


folder = Path(r"../data/OUTarea2\Aug")

dates_by_year = defaultdict(set)

for f in folder.glob("*.tif"):
    if f.name.startswith("S2B_MSI") or f.name.startswith("S2A_MSI") or f.name.startswith("S2C_MSI"):
        parts = f.name.split("_")
        year = parts[2]
        date = f"{parts[2]}-{parts[3]}-{parts[4]}"
        dates_by_year[year].add(date)

for year in sorted(dates_by_year):
    print(year, len(dates_by_year[year]))




In [ ]:

# Find possible red bands
redbands = glob.glob(os.path.join(data_folder, "*Rrs_665*.tif")) + \
           glob.glob(os.path.join(data_folder, "*Rrs_667*.tif"))

print(f"Found {len(redbands)} red band files")



In [ ]:

all_red = []
profile = None

for reds in redbands:

    # Match flag file
    flag_file = reds.replace("Rrs_665", "l2_flags").replace("Rrs_667", "l2_flags")

    if not os.path.exists(flag_file):
        print("Missing flags for:", os.path.basename(reds))
        continue

    # Read red band
    with rasterio.open(reds) as src:
        red = src.read(1).astype(np.float32)
        profile = src.profile

    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)

    # ACOLITE masks
    mask_out      = has_flag(flags, 4)
    mask_cirrus   = has_flag(flags, 1)
    mask_negative = has_flag(flags, 3)
    mask_toa      = has_flag(flags, 2)

    valid = ~(mask_out | mask_negative | mask_cirrus)

    # Apply mask
    red_masked = np.where(valid, red, np.nan)

    # Drop empty scenes
    if np.all(np.isnan(red_masked)):
        print(f"Dropping empty scene: {os.path.basename(reds)}")
        continue

    all_red.append(red_masked)

print(f"Kept {len(all_red)} valid scenes")

# -----------------------------
# STACK + COMPOSITE
# -----------------------------
if len(all_red) == 0:
    raise RuntimeError("No valid scenes after masking!")

red_stack = np.stack(all_red, axis=0)

# Monthly median
monthly_median_red = np.nanmedian(red_stack, axis=0)


valid_count = np.sum(~np.isnan(red_stack), axis=0)


# -----------------------------
# SAVE OUTPUT
# -----------------------------
monthly_red_path = os.path.join(data_folder, "monthly_median_red_masked.tif")


profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_red.shape[0],
    width=monthly_median_red.shape[1],
)

# Save median
with rasterio.open(monthly_red_path, "w", **profile) as dst:
    dst.write(monthly_median_red.astype(np.float32), 1)

# Save valid count map
profile.update(dtype=rasterio.int16, nodata=0)


print("Saved:", monthly_red_path)

In [ ]:
# Find possible green bands
greenbands = glob.glob(os.path.join(data_folder, "*Rrs_559*.tif")) + \
            glob.glob(os.path.join(data_folder, "*Rrs_561*.tif")) + \
            glob.glob(os.path.join(data_folder, "*Rrs_560*.tif"))

print(f"Found {len(greenbands)} green band files")



In [ ]:

all_green = []
profile = None

for greens in greenbands:

    flag_file = greens.replace("Rrs_559", "l2_flags").replace("Rrs_561", "l2_flags").replace("Rrs_560", "l2_flags")

    if not os.path.exists(flag_file):
        print(" Missing flags for:", os.path.basename(greens))
        continue

    with rasterio.open(greens) as src:
        green = src.read(1).astype(np.float32)
        profile = src.profile

    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)

    # ACOLITE masks
    mask_out      = has_flag(flags, 4)
    mask_cirrus   = has_flag(flags, 1)
    mask_negative = has_flag(flags, 3)
    mask_toa      = has_flag(flags, 2)

    
    valid = ~(mask_out | mask_negative | mask_cirrus)

    # Apply mask
    green_masked = np.where(valid, green, np.nan)

    # Drop empty scenes
    if np.all(np.isnan(green_masked)):
        print(f"Dropping empty scene: {os.path.basename(greens)}")
        continue

    all_green.append(green_masked)

print(f"Kept {len(all_green)} valid scenes")

# -----------------------------
# STACK + COMPOSITE
# -----------------------------
if len(all_green) == 0:
    raise RuntimeError("No valid scenes after masking!")

green_stack = np.stack(all_green, axis=0)

# Monthly median
monthly_median_green = np.nanmedian(green_stack, axis=0)

# Observation count (quality layer)
valid_count = np.sum(~np.isnan(green_stack), axis=0)


# -----------------------------
# SAVE OUTPUT
# -----------------------------
monthly_green_path = os.path.join(data_folder, "monthly_median_green_masked.tif")


profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_green.shape[0],
    width=monthly_median_green.shape[1],
)

# Save median
with rasterio.open(monthly_green_path, "w", **profile) as dst:
    dst.write(monthly_median_green.astype(np.float32), 1)



print("Saved:", monthly_green_path)


In [ ]:
bluebands = glob.glob(os.path.join(data_folder, "*Rrs_492*.tif")) + \
           glob.glob(os.path.join(data_folder, "*Rrs_489*.tif"))

print(f"Found {len(bluebands)} blue band files")



In [ ]:

all_blue = []
profile = None

for blues in bluebands:

    flag_file = blues.replace("Rrs_492", "l2_flags").replace("Rrs_489", "l2_flags")

    if not os.path.exists(flag_file):
        print(" Missing flags for:", os.path.basename(blues))
        continue

    with rasterio.open(blues) as src:
        blue = src.read(1).astype(np.float32)
        profile = src.profile

    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)

    # ACOLITE masks
    mask_out      = has_flag(flags, 4)
    mask_cirrus   = has_flag(flags, 1)
    mask_negative = has_flag(flags, 3)
    mask_toa      = has_flag(flags, 2)

    
    valid = ~(mask_out | mask_negative | mask_cirrus)

    # Apply mask
    blue_masked = np.where(valid, blue, np.nan)

    # Drop empty scenes
    if np.all(np.isnan(blue_masked)):
        print(f"Dropping empty scene: {os.path.basename(blues)}")
        continue

    all_blue.append(blue_masked)

print(f"Kept {len(all_blue)} valid scenes")

# -----------------------------
# STACK + COMPOSITE
# -----------------------------
if len(all_blue) == 0:
    raise RuntimeError("No valid scenes after masking!")

blue_stack = np.stack(all_blue, axis=0)

# Monthly median
monthly_median_blue = np.nanmedian(blue_stack, axis=0)

# Observation count (quality layer)
valid_count = np.sum(~np.isnan(blue_stack), axis=0)


# -----------------------------
# SAVE OUTPUT
# -----------------------------
monthly_blue_path = os.path.join(data_folder, "monthly_median_blue_masked.tif")


profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_blue.shape[0],
    width=monthly_median_blue.shape[1],
)

# Save median
with rasterio.open(monthly_blue_path, "w", **profile) as dst:
    dst.write(monthly_median_blue.astype(np.float32), 1)



print("Saved:", monthly_blue_path)

In [ ]:
# for NIR bands

NIRbands = glob.glob(os.path.join(data_folder, "*Rrs_865*.tif")) + \
        glob.glob(os.path.join(data_folder, "*Rrs_866*.tif")) + \
        glob.glob(os.path.join(data_folder, "*Rrs_864*.tif"))

print(f"Found {len(NIRbands)} NIR band files")


In [ ]:

all_NIR = []
profile = None

for NIRs in NIRbands:

    flag_file = NIRs.replace("Rrs_865", "l2_flags").replace("Rrs_866", "l2_flags").replace("Rrs_864", "l2_flags")

    if not os.path.exists(flag_file):
        print(" Missing flags for:", os.path.basename(NIRs))
        continue

    with rasterio.open(NIRs) as src:
        NIR = src.read(1).astype(np.float32)
        profile = src.profile

    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)

    # ACOLITE masks
    mask_out      = has_flag(flags, 4)
    mask_cirrus   = has_flag(flags, 1)
    mask_negative = has_flag(flags, 3)
    mask_toa      = has_flag(flags, 2)

    
    valid = ~(mask_out | mask_negative | mask_cirrus)

    # Apply mask
    NIR_masked = np.where(valid, NIR, np.nan)

    # Drop empty scenes
    if np.all(np.isnan(NIR_masked)):
        print(f"Dropping empty scene: {os.path.basename(NIRs)}")
        continue

    all_NIR.append(NIR_masked)

print(f"Kept {len(all_NIR)} valid scenes")

# -----------------------------
# STACK + COMPOSITE
# -----------------------------
if len(all_NIR) == 0:
    raise RuntimeError("No valid scenes after masking!")

NIR_stack = np.stack(all_NIR, axis=0)

# Monthly median
monthly_median_NIR = np.nanmedian(NIR_stack, axis=0)

# Observation count (quality layer)
valid_count = np.sum(~np.isnan(NIR_stack), axis=0)


# -----------------------------
# SAVE OUTPUT
# -----------------------------
monthly_NIR_path = os.path.join(data_folder, "monthly_median_NIR_masked.tif")


profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_NIR.shape[0],
    width=monthly_median_NIR.shape[1],
)

# Save median
with rasterio.open(monthly_NIR_path, "w", **profile) as dst:
    dst.write(monthly_median_NIR.astype(np.float32), 1)


print("Saved:", monthly_NIR_path)

In [ ]:
# getting the land mask
blue  = rasterio.open(r"../data/OUTarea1\Feb\monthly_median_blue_masked.tif").read(1)
green = rasterio.open(r"../data/OUTarea1\Feb\monthly_median_green_masked.tif").read(1)
red   = rasterio.open(r"../data/OUTarea1\Feb\monthly_median_red_masked.tif").read(1)
nir   = rasterio.open(r"../data/OUTarea1\Feb\monthly_median_NIR_masked.tif").read(1)

stack = np.stack([blue, green, red, nir])

rho_max = np.nanmax(stack, axis=0)
rho_min = np.nanmin(stack, axis=0)

# 1) Bright pixels
mask_bright = rho_max > 0.20

# 2) NIR peak
mask_nirpeak = rho_max < nir

# 3) Non-water ratio
mask_nonwater = (nir / red > 0.9) & (nir > 0.05)

# 4) Spectral flatness
mask_white = (rho_max - rho_min) / rho_max < 0.2

# Combine land mask
land_mask = mask_bright | mask_nirpeak | mask_nonwater | mask_white

#with rasterio.open(land_mask, "w", ) as dst:
 #   dst.write(land_mask.astype(np.float32), 1)

In [ ]:
from scipy import ndimage as nd
# ndimage.distance_transform_edt(input_array)

pixel_size = 10
dist = nd.distance_transform_edt(1-land_mask ) * pixel_size
coastal_mask = dist < 100

#with rasterio.open(r"../data/OUTarea2\Jan\monthly_median_spm_masked.tif") as src:
 #   spm = src.read(1)

#spm_clean = np.where(coastal_mask, np.nan, spm)

In [ ]:
spm_files = glob.glob(os.path.join(data_folder, "*SPM_Novoa2017*.tif"))

print(f"Found {len(spm_files)} spm novoa files")

In [ ]:
# updating the spm with limited scenes


all_spm_data = []

for spm_file in spm_files:
    
    # Find matching flag file
    flag_file = spm_file.replace("SPM_Novoa2017", "l2_flags")
    
    if not os.path.exists(flag_file):
        print("Missing flags for:", spm_file)
        continue
    
    # Read SPM
    with rasterio.open(spm_file) as src:
        spm = src.read(1).astype(np.float32)
        profile = src.profile
    
    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)
    
    # -----------------------------
    # ACOLITE FLAG MASKING
    # -----------------------------
    
    mask_out = has_flag(flags, 4)
    mask_cirrus  = has_flag(flags, 1)
    mask_swir = has_flag(flags, 0)
    mask_negative  = has_flag(flags, 3)
    mask_toa   = has_flag(flags, 2)

    valid = ~(mask_out |  mask_toa  | mask_negative | mask_cirrus | mask_swir)
    
    
    spm_masked = np.where(valid, spm, np.nan)
    #including the land mask for the spm calculation

    spm_water = np.where(land_mask, np.nan, spm_masked)
   
    if np.all(np.isnan(spm_water)):
        print(f" Dropping empty scene: {os.path.basename(spm_file)}")
        continue

    all_spm_data.append(spm_water)

print(f"Kept {len(all_spm_data)} scenes out of {len(spm_files)}")

# ---- STACK ----
if len(all_spm_data) == 0:
    raise RuntimeError("No valid scenes after masking!")

spm_stack = np.stack(all_spm_data, axis=0)

# ---- MONTHLY MEDIAN ----
monthly_median_spm = np.nanmedian(spm_stack, axis=0)

# ---- SAVE ----
monthly_spm_path = os.path.join(data_folder, "monthly_median_spm_masked.tif")

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_spm.shape[0],
    width=monthly_median_spm.shape[1],
)

with rasterio.open(monthly_spm_path, 'w', **profile) as dst:
    dst.write(monthly_median_spm.astype(np.float32), 1)

print("Saved:", monthly_spm_path)

In [ ]:
#plot updated spm land masked based on reflectance
with rasterio.open(r"../data/OUTarea1\Feb\monthly_median_spm_masked.tif") as src:
     monthly_median_spm = src.read(1)
#outpng = "area2JUNSPMmedian"
plt.figure(figsize=(8,6))
plt.imshow(monthly_median_spm,  vmax=100, cmap="turbo")
plt.colorbar(label="SPM (g m⁻3)")
plt.title("Median SPM for Feb")
#plt.savefig(outpng, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# get maximum monthly spm

spm_files = glob.glob(os.path.join(data_folder, "*SPM_Novoa2017*.tif"))

all_spm_data = []

for spm_file in spm_files:
    
    # Find matching flag file
    flag_file = spm_file.replace("SPM_Novoa2017", "l2_flags")
    
    if not os.path.exists(flag_file):
        print("Missing flags for:", spm_file)
        continue
    
    # Read SPM
    with rasterio.open(spm_file) as src:
        spm = src.read(1).astype(np.float32)
        profile = src.profile
    
    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)
    
    # -----------------------------
    # ACOLITE FLAG MASKING
    # -----------------------------
    
    mask_out = has_flag(flags, 4)
    mask_cirrus  = has_flag(flags, 1)
    mask_swir = has_flag(flags, 0)
    mask_negative  = has_flag(flags, 3)
    mask_toa   = has_flag(flags, 2)

    valid = ~(mask_out |  mask_toa  | mask_negative | mask_cirrus | mask_swir)
    
    
    spm_masked = np.where(valid, spm, np.nan)
    #including the land mask for the spm calculation

    spm_water = np.where(land_mask, np.nan, spm_masked)
   
    if np.all(np.isnan(spm_water)):
        print(f" Dropping empty scene: {os.path.basename(spm_file)}")
        continue

    all_spm_data.append(spm_water)

print(f"Kept {len(all_spm_data)} scenes out of {len(spm_files)}")

# ---- STACK ----
if len(all_spm_data) == 0:
    raise RuntimeError("No valid scenes after masking!")

spm_stack = np.stack(all_spm_data, axis=0)

# ---- MONTHLY MAX ----
monthly_max_spm = np.nanmax(spm_stack, axis=0) # to get max value: np.nanmax(spm_stack, axis=0)to get percentile value np.nanpercentile(spm_stack, 90, axis=0)

# ---- SAVE ----
monthly_spm_path = os.path.join(data_folder, "monthly_max_spm_masked.tif")

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_max_spm.shape[0],
    width=monthly_max_spm.shape[1],
)

with rasterio.open(monthly_spm_path, 'w', **profile) as dst:
    dst.write(monthly_max_spm.astype(np.float32), 1)

print("Saved:", monthly_spm_path)

In [ ]:
with rasterio.open(r"../data/OUTarea2\Dec\monthly_max_spm_masked.tif") as src:
    monthly_max_spm = src.read(1)
#outpng="maxspmJunArea2"
plt.figure(figsize=(8,6))
plt.imshow(monthly_max_spm, vmax=40, cmap="turbo")
plt.colorbar(label="SPM (g m⁻3)")
plt.title("Max SPM for December")
#plt.savefig(outpng, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# hue angle composite
hue_files = glob.glob(os.path.join(data_folder, "*hue_angle*.tif"))
print(f"Found {len(hue_files)} hue files")

In [ ]:

all_hue_data = []

for hue_file in hue_files:
    
    # Find matching flag file
    flag_file = hue_file.replace("hue_angle", "l2_flags")
    
    if not os.path.exists(flag_file):
        print("Missing flags for:", hue_file)
        continue
    
    # Read SPM
    with rasterio.open(hue_file) as src:
        hue = src.read(1).astype(np.float32)
        profile = src.profile
    
    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)
    
    # -----------------------------
    # ACOLITE FLAG MASKING
    # -----------------------------
    
    mask_out = has_flag(flags, 4)
    mask_cirrus  = has_flag(flags, 1)
    mask_swir = has_flag(flags, 0)
    mask_negative  = has_flag(flags, 3)
    mask_toa   = has_flag(flags, 2)

    valid = ~(mask_out |  mask_toa  | mask_negative | mask_cirrus)
    
    
    hue_masked = np.where(valid, hue, np.nan)
    #including the land mask for the spm calculation

    hue_water = np.where(land_mask, np.nan, hue_masked)
   
    if np.all(np.isnan(hue_water)):
        print(f" Dropping empty scene: {os.path.basename(hue_file)}")
        continue

    all_hue_data.append(hue_water)

print(f"Kept {len(all_hue_data)} scenes out of {len(hue_files)}")

# ---- STACK ----
if len(all_hue_data) == 0:
    raise RuntimeError("No valid scenes after masking!")

hue_stack = np.stack(all_hue_data, axis=0)

# ---- MONTHLY MEDIAN ----
monthly_median_hue = np.nanmedian(hue_stack, axis=0)

# ---- SAVE ----
monthly_hue_path = os.path.join(data_folder, "monthly_median_hue_masked.tif")

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_hue.shape[0],
    width=monthly_median_hue.shape[1],
)

with rasterio.open(monthly_hue_path, 'w', **profile) as dst:
    dst.write(monthly_median_hue.astype(np.float32), 1)

print("Saved:", monthly_hue_path)

In [ ]:
hue = r"../data/OUTarea2\Dec\monthly_median_hue_masked.tif"

with rasterio.open(hue) as src:
    monthly_median_hue= src.read(1)
#outpng="hueangleJunarea2"
plt.figure(figsize=(8,6))
plt.imshow(monthly_median_hue, vmin=40, vmax=220)
plt.colorbar(label="Hue angle")
plt.title("Median hue angle December")
#plt.savefig(outpng, dpi=300, bbox_inches="tight")

plt.show()

#hue angle should follow the pattern of sediment more or less

In [ ]:
# generate slh as well, to get contribution of backscattering 

slh_files = glob.glob(os.path.join(data_folder, "*slh*.tif"))

print(f"Found {len(slh_files)} slh files")

In [ ]:

all_slh_data = []

for slh_file in slh_files:
    
    # Find matching flag file
    flag_file = slh_file.replace("slh", "l2_flags")
    
    if not os.path.exists(flag_file):
        print("Missing flags for:", slh_file)
        continue
    
    # Read slh
    with rasterio.open(slh_file) as src:
        slh = src.read(1).astype(np.float32)
        profile = src.profile
    
    # Read flags
    with rasterio.open(flag_file) as src:
        flags = src.read(1)
    
    # -----------------------------
    # ACOLITE FLAG MASKING
    # -----------------------------
    
    mask_out = has_flag(flags, 4)
    mask_cirrus  = has_flag(flags, 1)
    mask_swir = has_flag(flags, 0)
    mask_negative  = has_flag(flags, 3)
    mask_toa   = has_flag(flags, 2)

    valid = ~(mask_out |  mask_toa  | mask_negative | mask_cirrus)
    
    
    slh_masked = np.where(valid, slh, np.nan)
    #including the land mask for the spm calculation

    slh_water = np.where(land_mask, np.nan, slh_masked)
   
    if np.all(np.isnan(slh_water)):
        print(f" Dropping empty scene: {os.path.basename(slh_file)}")
        continue

    all_slh_data.append(slh_water)

print(f"Kept {len(all_slh_data)} scenes out of {len(slh_files)}")

# ---- STACK ----
if len(all_slh_data) == 0:
    raise RuntimeError("No valid scenes after masking!")

slh_stack = np.stack(all_slh_data, axis=0)

# ---- MONTHLY MEDIAN ----
monthly_median_slh = np.nanmedian(slh_stack, axis=0)

# ---- SAVE ----
monthly_slh_path = os.path.join(data_folder, "monthly_median_slh_masked.tif")

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    height=monthly_median_slh.shape[0],
    width=monthly_median_slh.shape[1],
)

with rasterio.open(monthly_slh_path, 'w', **profile) as dst:
    dst.write(monthly_median_slh.astype(np.float32), 1)

print("Saved:", monthly_slh_path)

In [ ]:
#slh = r"../data/OUTarea2\Jun\monthly_median_slh_masked.tif"

#with rasterio.open(slh) as src:
 #   monthly_median_slh= src.read(1)

    
#outpng="SLHJun2"
plt.figure(figsize=(8,6))
plt.imshow(monthly_median_slh)
plt.colorbar(label="SLH")
plt.title("Scattering Line Height for November (land masked)")
#plt.savefig(outpng, dpi=300, bbox_inches="tight")

plt.show()